<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

## 👁️ Week 11 — Introduction to Computer Vision: Five Tasks, One Library
### CP020003 Artificial Intelligence 2026 — In-Class Notebook

Last week we taught a model to read *numbers*. This week we teach it to read *pictures*.

Modern computer vision is not one problem — it's a family of related problems that all start the same way (a grid of pixels) and end very differently:

| Task | Question it answers | Output |
|---|---|---|
| 🏷️ Image Classification | "What is this?" | One label for the whole image |
| 📦 Object Detection (BBox) | "What objects, and where?" | Axis-aligned boxes + labels |
| 🔄 Oriented Object Detection (OBB) | "Where, **and at what angle**?" | Rotated boxes + labels |
| 🎨 Semantic / Instance Segmentation | "Which *pixels* belong to what?" | Per-pixel masks |
| 📏 Depth Estimation | "How far away is each pixel?" | A depth value per pixel |
| 🤸 Pose Estimation | "Where are the joints?" | A skeleton of keypoints per object |

We'll build a working, trainable, measurable pipeline for **all six** (bbox + OBB counted separately) using [Ultralytics](https://docs.ultralytics.com/) — the library behind the YOLO family of models — plus a pretrained depth model for the one task YOLO doesn't cover. Every section follows the same rhythm:

1. **Load a small public "toy" dataset** so training finishes in minutes on a free Colab GPU.
2. **📁 Guide: Use Your Own Dataset** — a markdown cheat-sheet showing exactly how to fold *your* project's images into the same pipeline.
3. **Train** a tiny model for a few epochs (just enough to prove the pipeline works).
4. **Run inference** on new images.
5. **Evaluate** with the metrics the industry actually uses for that task.

> 💡 **Runtime:** `Runtime → Change runtime type → T4 GPU` (free tier). Every training run below is capped at a handful of epochs on tiny (8–160 image) datasets on purpose — this class is about the *pipeline*, not squeezing out state-of-the-art accuracy. Swap in your own data and crank up `epochs` once you're ready.


## 0. Setup 🔧

Colab ships `torch` and `matplotlib` already. We add `ultralytics`, which bundles classification, detection, OBB, segmentation, and pose models behind one consistent API, plus `timm` (used by our pretrained depth model later).


In [ ]:
# Write the name of your computer vision library here

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

SEED = # Write your lucky number here
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("⚠️  No GPU detected — go to Runtime > Change runtime type > T4 GPU for a big speedup.")

import ultralytics
ultralytics.checks()

## 1. Image Classification 🏷️

**Task:** one label for the whole image. This is the "hello world" of computer vision — everything else we do today is a variation on the same idea (a CNN backbone extracting features), with a different "head" bolted on top.

### 1.1 Load a public toy dataset

Ultralytics ships a handful of tiny built-in datasets it will auto-download the first time you reference them. For classification we'll use **`mnist160`** — 160 images (a small slice of MNIST) already arranged into the folder structure Ultralytics expects. Nothing to download manually — just pass the name and training triggers the download.


In [ ]:
CLS_DATASET = # Write your dataset name here  # tiny built-in toy dataset (auto-downloads on first use)

# Peek at what Ultralytics will fetch — it caches under ~/datasets by default
from ultralytics.utils import DATASETS_DIR
print("Datasets cache dir:", DATASETS_DIR)

### 📁 Guide: Use Your Own Dataset (Classification)

Ultralytics classification expects a plain **"ImageFolder"** layout — one sub-folder per class, split into `train` and `val` (and optionally `test`):

```
my_classification_dataset/
├── train/
│   ├── cat/
│   │   ├── img001.jpg
│   │   └── img002.jpg
│   ├── dog/
│   │   ├── img101.jpg
│   │   └── img102.jpg
│   └── bird/
│       └── ...
└── val/
    ├── cat/
    ├── dog/
    └── bird/
```

To train on it, upload the folder to Colab (or mount Google Drive) and point `data=` at the **root folder** — no YAML file needed for classification:

```python
from ultralytics import YOLO
model = YOLO("yolov8n-cls.pt")
model.train(data="/content/my_classification_dataset", epochs=20, imgsz=224)
```

That's it — the class names are inferred automatically from the sub-folder names.


### 1.2 Train (quick, few epochs)

In [ ]:
cls_model = # Write your model name here   # smallest classification model, pretrained on ImageNet

cls_results = cls_model.train(
    data=CLS_DATASET,
    epochs=5,
    imgsz=64,        # MNIST digits are tiny — no need for a big image size
    batch=32,
    seed=SEED,
    verbose=False,
    plots=True,
)

### 1.3 Inference

In [ ]:
import glob
from pathlib import Path
from PIL import Image

# Recursively find a few sample images — don't assume exact folder depth/extension
cls_root = Path(DATASETS_DIR) / CLS_DATASET
sample_imgs = []
for ext in ("*.jpg", "*.jpeg", "*.png"):
    sample_imgs += list(cls_root.rglob(ext))
sample_imgs = sorted(sample_imgs)[:6]

print(f"Looking under: {cls_root}")
print(f"Found {len(sample_imgs)} sample images")
assert sample_imgs, f"No images found under {cls_root} — check that training actually downloaded the dataset."

preds = cls_model.predict([str(p) for p in sample_imgs], verbose=False)

fig, axes = plt.subplots(1, len(preds), figsize=(3 * len(preds), 3))
if len(preds) == 1:
    axes = [axes]
for ax, r in zip(axes, preds):
    img = Image.open(r.path)
    top1 = r.probs.top1
    conf = r.probs.top1conf.item()
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{r.names[top1]} ({conf:.1%})", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 1.4 Metrics

Classification is scored on **Top-1 accuracy** (is the single best guess correct?) and **Top-5 accuracy** (is the true label among the model's 5 best guesses?) — standard since the ImageNet benchmark.


In [ ]:
cls_metrics = cls_model.val(data=CLS_DATASET, split="test", verbose=False)

print(f"Top-1 accuracy: {cls_metrics.top1:.4f}")
print(f"Top-5 accuracy: {cls_metrics.top5:.4f}")
# A confusion matrix PNG is also saved automatically under cls_metrics.save_dir
print("Full results & confusion matrix saved to:", cls_metrics.save_dir)

## 2. Object Detection — Axis-Aligned Bounding Boxes 📦

**Task:** find every object of interest and draw an upright rectangle around it, with a class label and a confidence score. This is what powers things like self-checkout cameras, security systems, and most "count the cars in this parking lot" applications.

### 2.1 Load a public toy dataset

We use **`coco8`** — 8 images sampled from COCO, already split into 4 train / 4 val, with YOLO-format label files. It ships with a ready-made `coco8.yaml` config, so we just reference it by name.


In [ ]:
DET_DATASET = "coco8.yaml"   # 8-image toy slice of COCO, auto-downloads

### 📁 Guide: Use Your Own Dataset (Detection)

Detection needs **images + one `.txt` label file per image** (same filename, `.txt` extension) plus a small **YAML config** describing the folders and class names.

```
my_detection_dataset/
├── images/
│   ├── train/
│   │   ├── img001.jpg
│   │   └── img002.jpg
│   └── val/
│       └── img101.jpg
└── labels/
    ├── train/
    │   ├── img001.txt
    │   └── img002.txt
    └── val/
        └── img101.txt
```

Each label line is **one object per row**, normalized 0–1, in YOLO format:

```
<class_id> <x_center> <y_center> <width> <height>
```

e.g. `0 0.532 0.481 0.220 0.340` — class `0`, centered at 53.2%/48.1% of the image, spanning 22.0% of the width and 34.0% of the height.

Then write a `my_dataset.yaml`:

```yaml
path: /content/my_detection_dataset   # root folder
train: images/train
val: images/val
names:
  0: cat
  1: dog
  2: bird
```

```python
model = YOLO("yolov8n.pt")
model.train(data="/content/my_detection_dataset/my_dataset.yaml", epochs=50, imgsz=640)
```

> 🛠️ Labeling your own images? Tools like **Roboflow**, **CVAT**, or **LabelImg** export directly to this YOLO format.


### 2.2 Train (quick, few epochs)

In [ ]:
det_model = YOLO("yolov8n.pt")   # nano detection model, pretrained on full COCO

det_results = det_model.train(
    data=DET_DATASET,
    epochs=20,
    imgsz=640,
    batch=4,
    seed=SEED,
    verbose=False,
    plots=True,
)

### 2.3 Inference

In [ ]:
det_preds = det_model.predict(f"{DATASETS_DIR}/coco8/images/val", conf=0.25, verbose=False)

fig, axes = plt.subplots(1, len(det_preds), figsize=(5 * len(det_preds), 5))
if len(det_preds) == 1:
    axes = [axes]
for ax, r in zip(axes, det_preds):
    ax.imshow(r.plot()[..., ::-1])   # r.plot() returns BGR — flip to RGB for matplotlib
    ax.axis("off")
plt.tight_layout()
plt.show()

### 2.4 Metrics

Detection quality is reported per-class and as an overall summary:

- **Precision / Recall** at a chosen confidence threshold
- **mAP50** — mean Average Precision at IoU ≥ 0.50 (a "loose" match)
- **mAP50-95** — mAP averaged over IoU thresholds 0.50 → 0.95 (the strict, COCO-standard number)


In [ ]:
det_metrics = det_model.val(data=DET_DATASET, verbose=False)

print(f"Precision:  {det_metrics.box.mp:.4f}")
print(f"Recall:     {det_metrics.box.mr:.4f}")
print(f"mAP50:      {det_metrics.box.map50:.4f}")
print(f"mAP50-95:   {det_metrics.box.map:.4f}")
print("\nPer-class mAP50-95:")
for i, name in det_metrics.names.items():
    print(f"  {name:<15s} {det_metrics.box.maps[i]:.4f}")
print("\nConfusion matrix & PR curves saved to:", det_metrics.save_dir)

## 3. Oriented Object Detection (OBB) 🔄

**Task:** same idea as detection, but the box can **rotate** to hug the object tightly. Axis-aligned boxes waste a lot of area (and hurt downstream tasks like counting or measuring) when objects sit at an angle — think aerial/satellite imagery of ships, planes, or parked cars, or a photo of a rotated document.

An OBB is usually described by either `(x_center, y_center, w, h, angle)` or the 4 corner points directly; Ultralytics uses **4 corner points** internally.

### 3.1 Load a public toy dataset

We use **`dota8`** — 8 images sampled from the DOTA aerial-imagery dataset, in OBB format, again with a ready-made YAML.


In [ ]:
OBB_DATASET = "dota8.yaml"   # 8-image toy slice of DOTA (aerial imagery), auto-downloads

### 📁 Guide: Use Your Own Dataset (OBB)

Folder layout is identical to detection (`images/train`, `images/val`, `labels/train`, `labels/val` + a YAML), but each label row has **8 coordinates** (the 4 corners, normalized 0–1) instead of a center + width/height:

```
<class_id> <x1> <y1> <x2> <y2> <x3> <y3> <x4> <y4>
```

Corners are listed in order (e.g. clockwise from top-left). Annotation tools like **Roboflow** and **CVAT** both support drawing rotated boxes and exporting in this format.

```python
model = YOLO("yolov8n-obb.pt")
model.train(data="/content/my_obb_dataset/my_dataset.yaml", epochs=50, imgsz=640)
```


### 3.2 Train (quick, few epochs)

In [ ]:
obb_model = # Write your model name here   # nano OBB model, pretrained on DOTA

obb_results = obb_model.train(
    data=OBB_DATASET,
    epochs=20,
    imgsz=640,
    batch=4,
    seed=SEED,
    verbose=False,
    plots=True,
)

### 3.3 Inference

In [ ]:
obb_preds = obb_model.predict(f"{DATASETS_DIR}/dota8/images/val", conf=0.25, verbose=False)

fig, axes = plt.subplots(1, len(obb_preds), figsize=(5 * len(obb_preds), 5))
if len(obb_preds) == 1:
    axes = [axes]
for ax, r in zip(axes, obb_preds):
    ax.imshow(r.plot()[..., ::-1])
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3.4 Metrics

Same family of metrics as bounding-box detection, but IoU is computed between **rotated** polygons instead of axis-aligned rectangles — a harder geometric problem, which is part of why OBB models exist as their own model family rather than a post-processing trick on top of regular detection.


In [ ]:
obb_metrics = obb_model.val(data=OBB_DATASET, verbose=False)

print(f"Precision:  {obb_metrics.box.mp:.4f}")
print(f"Recall:     {obb_metrics.box.mr:.4f}")
print(f"mAP50:      {obb_metrics.box.map50:.4f}")
print(f"mAP50-95:   {obb_metrics.box.map:.4f}")
print("\nResults saved to:", obb_metrics.save_dir)

## 4. Segmentation 🎨

**Task:** instead of a box, predict a **pixel-level mask** for every object — the outline hugs the object's exact silhouette. (Ultralytics' segmentation head produces *instance* masks — one mask per object instance; merging all instances of the same class gives you the *semantic* segmentation map. We'll show both views below.)

### 4.1 Load a public toy dataset

We use **`coco8-seg`** — the same 8 COCO images as before, with polygon mask annotations instead of just boxes.


In [ ]:
SEG_DATASET = "coco8-seg.yaml"   # 8-image toy slice of COCO with masks, auto-downloads

### 📁 Guide: Use Your Own Dataset (Segmentation)

Same `images/` + `labels/` + YAML layout again. Each label row is a **variable-length polygon** (normalized 0–1 x,y pairs tracing the object's outline) instead of a fixed-size box:

```
<class_id> <x1> <y1> <x2> <y2> <x3> <y3> ... <xn> <yn>
```

```python
model = YOLO("yolov8n-seg.pt")
model.train(data="/content/my_seg_dataset/my_dataset.yaml", epochs=50, imgsz=640)
```

> 🛠️ Polygon labeling is slower than boxes — **Roboflow**, **CVAT**, and **Segment Anything (SAM)**-assisted tools (Ultralytics even ships a `SAM` auto-labeler, see `ultralytics.SAM`) can speed this up a lot.


### 4.2 Train (quick, few epochs)

In [ ]:
seg_model = # Write your model name here   # nano segmentation model, pretrained on COCO

seg_results = seg_model.train(
    data=SEG_DATASET,
    epochs=20,
    imgsz=640,
    batch=4,
    seed=SEED,
    verbose=False,
    plots=True,
)

### 4.3 Inference

In [ ]:
seg_preds = seg_model.predict(f"{DATASETS_DIR}/coco8-seg/images/val", conf=0.25, verbose=False)

fig, axes = plt.subplots(1, len(seg_preds), figsize=(5 * len(seg_preds), 5))
if len(seg_preds) == 1:
    axes = [axes]
for ax, r in zip(axes, seg_preds):
    ax.imshow(r.plot()[..., ::-1])   # instance masks overlaid with boxes
    ax.axis("off")
plt.tight_layout()
plt.show()

### 4.4 Metrics

Segmentation gets **two** sets of the detection metrics — one for the bounding boxes, one for the masks themselves (mask mAP uses pixel-wise IoU between predicted and ground-truth masks instead of box IoU):


In [ ]:
seg_metrics = seg_model.val(data=SEG_DATASET, verbose=False)

print("Box metrics:")
print(f"  mAP50:    {seg_metrics.box.map50:.4f}")
print(f"  mAP50-95: {seg_metrics.box.map:.4f}")

print("\nMask metrics:")
print(f"  mAP50:    {seg_metrics.seg.map50:.4f}")
print(f"  mAP50-95: {seg_metrics.seg.map:.4f}")
print("\nResults saved to:", seg_metrics.save_dir)

## 5. Monocular Depth Estimation 📏

**Task:** given a single 2D image, predict how far away every pixel is from the camera — no stereo pair, no LiDAR, just one photo. This is what powers portrait "blur the background" effects, robot navigation, and AR occlusion.

> ⚠️ **A note on tooling:** Ultralytics/YOLO's model family covers classification, detection, OBB, segmentation, and pose — but **not** depth estimation, so there's no `yolov8n-depth.pt` to reach for. Depth models use a different architecture (a dense encoder-decoder that outputs one value per pixel, e.g. **MiDaS** or **DPT**). We'll use a small **pretrained MiDaS** model for inference here (great zero-shot results with no training needed), and describe how you'd set up training data if you wanted to fine-tune one yourself.

### 5.1 Load a pretrained depth model


In [ ]:
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.to(device).eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.small_transform
print("MiDaS small loaded.")

### 📁 Guide: Use Your Own Dataset (Depth — if you want to fine-tune)

Depth training data is **RGB + a matching depth map** per image — either a real sensor reading (e.g. a LiDAR/structured-light depth map from datasets like **NYU Depth V2** or **KITTI**) or a rendered ground-truth from synthetic data:

```
my_depth_dataset/
├── rgb/
│   ├── img001.png
│   └── img002.png
└── depth/
    ├── img001.png     # 16-bit PNG, pixel value = depth (mm or a fixed scale)
    └── img002.png
```

Fine-tuning MiDaS/DPT isn't a one-liner like `model.train(...)` — you'd write a small PyTorch training loop that feeds `(rgb, depth)` pairs through the model and minimizes a scale-invariant loss (e.g. `SiLog` or a masked L1 loss, since ground-truth depth often has invalid/missing pixels). This is more advanced than a YOLO `.train()` call — for a class project, zero-shot MiDaS inference (below) is usually enough.


### 5.2 Inference

Let's run MiDaS on a couple of the images we already downloaded for the detection task, so we can compare "what's in the scene" (from Section 2) with "how far away is it".


In [ ]:
import cv2

depth_sample_paths = glob.glob(f"{DATASETS_DIR}/coco8/images/val/*.jpg")[:3]

fig, axes = plt.subplots(2, len(depth_sample_paths), figsize=(5 * len(depth_sample_paths), 8))

depth_maps = []
for i, p in enumerate(depth_sample_paths):
    img_bgr = cv2.imread(p)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_rgb.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    depth_map = prediction.cpu().numpy()
    depth_maps.append(depth_map)

    axes[0, i].imshow(img_rgb)
    axes[0, i].set_title("Input")
    axes[0, i].axis("off")

    im = axes[1, i].imshow(depth_map, cmap="inferno")
    axes[1, i].set_title("Predicted relative depth\n(brighter = closer)")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

### 5.3 Metrics

Monocular depth is judged against ground-truth depth maps with a standard set of metrics (lower is better for the error metrics, higher is better for accuracy):

- **AbsRel** — mean(|pred − gt| / gt), the average relative error
- **RMSE** — root-mean-squared error in depth units
- **δ1** — fraction of pixels where `max(pred/gt, gt/pred) < 1.25` (a tolerant "close enough" accuracy)

MiDaS predicts **relative, unscaled** depth (no metric units), so a real evaluation first fits a scale+shift to the ground truth before comparing — a detail every monocular-depth paper handles this way. Since our toy images don't ship LiDAR ground truth, we demonstrate the metric *functions* on a synthetic example so you know exactly what to compute once you have real (image, depth) pairs (e.g. from NYU Depth V2 or KITTI):


In [ ]:
def depth_metrics(pred, gt, eps=1e-6):
    # AbsRel, RMSE, and delta1 for monocular depth. pred/gt: same-shape numpy arrays.
    pred, gt = pred.astype(np.float64), gt.astype(np.float64)
    valid = gt > eps
    pred, gt = pred[valid], gt[valid]

    abs_rel = np.mean(np.abs(pred - gt) / gt)
    rmse = np.sqrt(np.mean((pred - gt) ** 2))
    ratio = np.maximum(pred / gt, gt / pred)
    delta1 = np.mean(ratio < 1.25)
    return {"AbsRel": abs_rel, "RMSE": rmse, "delta1": delta1}

# Synthetic demo: pretend this is (prediction, ground_truth) depth in meters
rng = np.random.default_rng(SEED)
gt_demo = rng.uniform(1.0, 10.0, size=(64, 64))
pred_demo = gt_demo + rng.normal(0, 0.4, size=gt_demo.shape)   # a "pretty good" model

print("Example on synthetic ground truth (replace with real NYU/KITTI pairs):")
for k, v in depth_metrics(pred_demo, gt_demo).items():
    print(f"  {k}: {v:.4f}")

## 6. Pose Estimation 🤸

**Task:** find each person (or animal) in the image **and** locate their skeleton — a fixed set of keypoints (nose, shoulders, elbows, wrists, hips, knees, ankles, ...) plus the visible/occluded state of each. This is what powers fitness apps, sports analytics, and motion-capture-lite.

### 6.1 Load a public toy dataset

We use **`coco8-pose`** — the same 8-image COCO slice, this time with the standard 17-keypoint human skeleton annotated.


In [ ]:
POSE_DATASET = "coco8-pose.yaml"   # 8-image toy slice of COCO with keypoints, auto-downloads

### 📁 Guide: Use Your Own Dataset (Pose)

Same `images/` + `labels/` + YAML layout, but each label row extends the detection format with **3 numbers per keypoint** (`x, y, visibility`):

```
<class_id> <x_center> <y_center> <w> <h> <x1> <y1> <v1> <x2> <y2> <v2> ... <xk> <yk> <vk>
```

`visibility`: `0` = not labeled, `1` = labeled but occluded, `2` = labeled and visible. Your YAML must also declare `kpt_shape: [num_keypoints, 3]` and, optionally, `flip_idx` (which keypoints swap under a horizontal flip — e.g. left/right wrist — so augmentation doesn't mirror your skeleton incorrectly):

```yaml
path: /content/my_pose_dataset
train: images/train
val: images/val
kpt_shape: [17, 3]
flip_idx: [0,2,1,4,3,6,5,8,7,10,9,12,11,14,13,16,15]
names:
  0: person
```

```python
model = YOLO("yolov8n-pose.pt")
model.train(data="/content/my_pose_dataset/my_dataset.yaml", epochs=50, imgsz=640)
```


### 6.2 Train (quick, few epochs)

In [ ]:
pose_model = # Write your model name here   # nano pose model, pretrained on COCO-pose

pose_results = pose_model.train(
    data=POSE_DATASET,
    epochs=20,
    imgsz=640,
    batch=4,
    seed=SEED,
    verbose=False,
    plots=True,
)

### 6.3 Inference

In [ ]:
pose_preds = pose_model.predict(f"{DATASETS_DIR}/coco8-pose/images/val", conf=0.25, verbose=False)

fig, axes = plt.subplots(1, len(pose_preds), figsize=(5 * len(pose_preds), 5))
if len(pose_preds) == 1:
    axes = [axes]
for ax, r in zip(axes, pose_preds):
    ax.imshow(r.plot()[..., ::-1])   # skeleton overlaid on the detected person
    ax.axis("off")
plt.tight_layout()
plt.show()

### 6.4 Metrics

Pose estimation reuses the **box** detection metrics for "did we find the person at all", plus a pose-specific mAP computed with **OKS** (Object Keypoint Similarity) in place of box IoU — OKS tolerates a little more positional error on naturally-wobbly keypoints (like the tip of a nose) than on stable ones (like a hip).


In [ ]:
pose_metrics = pose_model.val(data=POSE_DATASET, verbose=False)

print("Box metrics (person detection):")
print(f"  mAP50:    {pose_metrics.box.map50:.4f}")
print(f"  mAP50-95: {pose_metrics.box.map:.4f}")

print("\nPose (OKS) metrics:")
print(f"  mAP50:    {pose_metrics.pose.map50:.4f}")
print(f"  mAP50-95: {pose_metrics.pose.map:.4f}")
print("\nResults saved to:", pose_metrics.save_dir)

## 7. Wrap-Up 🎓

| Task | Model | Toy dataset | Label format | Headline metric |
|---|---|---|---|---|
| Classification | `yolov8n-cls.pt` | `mnist160` | `class_folder/image.jpg` | Top-1 / Top-5 accuracy |
| Detection (BBox) | `yolov8n.pt` | `coco8.yaml` | `class x_c y_c w h` | mAP50 / mAP50-95 |
| OBB | `yolov8n-obb.pt` | `dota8.yaml` | `class x1 y1 x2 y2 x3 y3 x4 y4` | mAP50 / mAP50-95 (rotated IoU) |
| Segmentation | `yolov8n-seg.pt` | `coco8-seg.yaml` | `class x1 y1 x2 y2 ... xn yn` | mask mAP50 / mAP50-95 |
| Depth Estimation | pretrained MiDaS | — (zero-shot) | RGB ↔ depth-map pairs | AbsRel, RMSE, δ1 |
| Pose | `yolov8n-pose.pt` | `coco8-pose.yaml` | box + `(x,y,v)` per keypoint | pose mAP50 / mAP50-95 (OKS) |

**The one pattern that repeats everywhere:**
```python
model = YOLO("<task-model>.pt")             # load a pretrained backbone
model.train(data="<dataset>.yaml", ...)     # fine-tune on your data
model.predict("<image or folder>")          # run inference
model.val(data="<dataset>.yaml")            # get the metrics that matter for this task
```
Once that pattern clicks, swapping in a new dataset — or a new task entirely — is mostly a matter of getting your **labels** into the right format.

---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 11 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>